<!-- notebook-header -->
# SQL e APIs para Coleta de Dados em ML

**Modulo:** 02 - Data Science  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** SQL, joins, window functions, REST APIs, paginacao, retry, JSON e scraping basico.


# SQL e APIs para Coleta de Dados em ML

Neste notebook, voce vai aprender a **buscar dados de fontes reais**: bancos de dados SQL,
APIs REST e web scraping. Em producao, dados nao vem em CSVs bonitos - vem de bancos
relacionais, endpoints JSON e paginas HTML.

**Analogia**: Se os notebooks anteriores ensinaram a cozinhar (Pandas, EDA), este ensina
a ir ao mercado. Nao adianta saber preparar o prato se voce nao sabe onde comprar
os ingredientes.

## Pre-requisitos e Fio Narrativo

| Conceito | Notebook | Por que |
|----------|----------|--------|
| Pandas DataFrames | `2_1_python_data_science` | Destino final dos dados |
| EDA | `2_2_eda_completa` | Saber o que procurar |

**Fio narrativo**: Em `2_1` voce aprendeu a manipular dados *ja carregados*. Em `2_2`,
aprendeu a *explorar*. Aqui voce aprende a *buscar* os dados na fonte. O pipeline completo
e: buscar (este notebook) -> explorar (`2_2`) -> transformar (`3_1`) -> modelar (`4_1`).

**Tempo estimado**: 8-10 horas

## Por que SQL e APIs em ML?

Na industria, dados de ML **nunca** vem de CSVs locais:

- **80% dos dados de empresas** estao em bancos SQL (PostgreSQL, MySQL, BigQuery)
- **APIs REST** sao a interface padrao para dados externos (clima, mercado, redes sociais)
- **Feature stores** modernos usam SQL como linguagem de consulta

Sem saber SQL e APIs, voce depende de outra pessoa para extrair seus dados - e isso
e um gargalo critico em projetos reais.

In [ ]:
import numpy as np
import pandas as pd
import sqlite3
import requests
import json
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print('=== SQL E APIS PARA ML ===' )
print(f'\nFontes de dados:')
print(f'  1. Bancos de dados SQL (PostgreSQL, MySQL, SQLite)')
print(f'  2. REST APIs (JSON over HTTP)')
print(f'  3. Web scraping (HTML parsing)')
print(f'  4. Arquivos estruturados (CSV, Parquet, JSON)')
print(f'  5. Data warehouses (BigQuery, Redshift)')


## 1. SQL Basico com SQLite

**Analogia**: SQL e como perguntar a um bibliotecario "quero todos os livros de ficcao
publicados depois de 2020, ordenados por avaliacao". O bibliotecario (banco de dados)
faz a busca e te entrega so o que voce pediu. Em Python puro, voce teria que ler todos
os livros da biblioteca e filtrar manualmente.

**Definicao formal**: SQL (Structured Query Language) e uma linguagem declarativa para
manipular dados em bancos relacionais. Os comandos fundamentais sao: SELECT (buscar),
WHERE (filtrar), GROUP BY (agregar), ORDER BY (ordenar), JOIN (combinar tabelas).

### Por que em ML?

SQL permite filtrar e agregar *no banco*, trazendo para Python apenas o resultado final.
Com 1 bilhao de linhas, carregar tudo em Pandas e impossivel; SQL resolve isso.

In [ ]:
# Criar banco de dados em memória
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print('\n=== SQL BÁSICO ===' )

# Criar tabela
cursor.execute('''
    CREATE TABLE users (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        age INTEGER,
        email TEXT,
        signup_date DATE
    )
''')

# Inserir dados
data = [
    (1, 'Alice', 25, 'alice@example.com', '2023-01-15'),
    (2, 'Bob', 30, 'bob@example.com', '2023-02-20'),
    (3, 'Charlie', 28, 'charlie@example.com', '2023-01-25'),
    (4, 'Diana', 35, 'diana@example.com', '2023-03-10'),
    (5, 'Eve', 26, 'eve@example.com', '2023-02-01')
]

cursor.executemany('INSERT INTO users VALUES (?, ?, ?, ?, ?)', data)
conn.commit()

print('\n1. SELECT simples:')
df = pd.read_sql_query('SELECT * FROM users', conn)
print(df)

print('\n2. Filtragem (WHERE):')
df_filtered = pd.read_sql_query('SELECT name, age FROM users WHERE age > 27', conn)
print(df_filtered)

print('\n3. Agregação (GROUP BY):')
cursor.execute('SELECT COUNT(*) as total, AVG(age) as media_idade FROM users')
result = cursor.fetchall()
print(f'Total: {result[0][0]}, Média de idade: {result[0][1]:.1f}')

print('\n4. Ordenação e Limite:')
df_sorted = pd.read_sql_query('SELECT name, age FROM users ORDER BY age DESC LIMIT 3', conn)
print(df_sorted)


### O que observar

- `pd.read_sql_query()` executa SQL e retorna DataFrame direto - e a ponte entre banco e Pandas
- WHERE filtra *antes* de trazer para Python - fundamental para tabelas grandes
- GROUP BY + COUNT/AVG/SUM sao as operacoes mais comuns em feature engineering via SQL

### O que concluir

- **SQL nao e alternativo a Pandas, e complementar**: SQL filtra e agrega no banco; Pandas faz o restante em memoria
- **Sempre teste com LIMIT primeiro**: `SELECT * FROM tabela_enorme` sem LIMIT pode travar tudo
- **SQLite e ideal para prototipagem**: roda em memoria, sem instalacao, mas nao escala para producao

### Conexao com outros notebooks

- O DataFrame resultante de SQL e o input direto para EDA em `2_2_eda_completa`
- Feature engineering via SQL e mais eficiente que via Pandas para dados grandes (ver `3_1_feature_engineering`)

## 2. SQL Avancado para ML

**Analogia**: SQL basico e pedir "todos os livros de ficcao". SQL avancado e pedir
"para cada autor, o livro com melhor avaliacao, combinado com dados de vendas de outra
tabela, rankeado por receita". JOINs e window functions sao as ferramentas que fazem isso.

**Definicao formal**: JOINs combinam tabelas por chave. Window functions (ROW_NUMBER, LAG,
SUM OVER) calculam agregacoes *sem colapsar linhas*, permitindo rankings, medias moveis e
features temporais diretamente em SQL.

### Por que em ML?

Feature engineering em SQL e 10-100x mais rapido que em Pandas para dados grandes. Window
functions permitem criar features como "rank do usuario dentro do segmento" ou "media movel
de compras dos ultimos 7 dias" direto na query.

In [ ]:
# SQL Avancado para ML
print('=== SQL AVANCADO ===')

# Criar tabela de eventos para demonstrar JOIN e Window Functions
cursor.execute('''
    CREATE TABLE events (
        event_id INTEGER PRIMARY KEY,
        user_id INTEGER,
        event_type TEXT,
        event_date DATE,
        value REAL,
        FOREIGN KEY (user_id) REFERENCES users(id)
    )
''')

events_data = [
    (1, 1, 'login', '2023-01-15', 0),
    (2, 1, 'purchase', '2023-01-16', 50.0),
    (3, 1, 'login', '2023-01-20', 0),
    (4, 2, 'login', '2023-02-01', 0),
    (5, 2, 'purchase', '2023-02-05', 120.0),
    (6, 2, 'purchase', '2023-02-10', 80.0),
    (7, 3, 'login', '2023-01-25', 0),
    (8, 4, 'login', '2023-03-01', 0),
    (9, 4, 'purchase', '2023-03-05', 200.0),
    (10, 5, 'login', '2023-02-01', 0),
]
cursor.executemany('INSERT INTO events VALUES (?, ?, ?, ?, ?)', events_data)
conn.commit()

# 1. JOIN: combinar usuarios com seus eventos
print('\n1. LEFT JOIN (usuarios + eventos):')
df_join = pd.read_sql_query('''
    SELECT u.name, u.age, COUNT(e.event_id) as total_events,
           SUM(CASE WHEN e.event_type = 'purchase' THEN 1 ELSE 0 END) as total_purchases,
           COALESCE(SUM(e.value), 0) as total_spent
    FROM users u
    LEFT JOIN events e ON u.id = e.user_id
    GROUP BY u.id, u.name, u.age
''', conn)
print(df_join)

# 2. Subquery: usuarios que gastaram acima da media
print('\n2. Subquery (gastaram acima da media):')
df_above_avg = pd.read_sql_query('''
    SELECT name, total_spent FROM (
        SELECT u.name, COALESCE(SUM(e.value), 0) as total_spent
        FROM users u
        LEFT JOIN events e ON u.id = e.user_id
        GROUP BY u.id, u.name
    ) WHERE total_spent > (
        SELECT AVG(value) FROM events WHERE value > 0
    )
''', conn)
print(df_above_avg)

# 3. Window Function: ranking por gasto
print('\n3. Window Function (ranking por gasto):')
df_rank = pd.read_sql_query('''
    SELECT u.name,
           COALESCE(SUM(e.value), 0) as total_spent,
           RANK() OVER (ORDER BY COALESCE(SUM(e.value), 0) DESC) as rank_gasto
    FROM users u
    LEFT JOIN events e ON u.id = e.user_id
    GROUP BY u.id, u.name
''', conn)
print(df_rank)

# 4. Feature engineering em SQL
print('\n4. Features para ML geradas em SQL:')
df_features = pd.read_sql_query('''
    SELECT u.id, u.name, u.age,
           COUNT(e.event_id) as n_events,
           SUM(CASE WHEN e.event_type = 'purchase' THEN 1 ELSE 0 END) as n_purchases,
           SUM(CASE WHEN e.event_type = 'login' THEN 1 ELSE 0 END) as n_logins,
           COALESCE(SUM(e.value), 0) as total_spent,
           COALESCE(AVG(CASE WHEN e.value > 0 THEN e.value END), 0) as avg_purchase
    FROM users u
    LEFT JOIN events e ON u.id = e.user_id
    GROUP BY u.id, u.name, u.age
''', conn)
print(df_features)
print('\n-> DataFrame pronto para modelagem, gerado 100% em SQL!')

### O que observar

- LEFT JOIN mantem TAREFA DO ALUNOS os usuarios, mesmo os sem eventos (Eve tem 1 login, 0 compras)
- COALESCE(SUM(...), 0) trata NULLs de usuarios sem eventos - sem isso, o resultado seria NULL
- RANK() OVER ordena por gasto sem colapsar as linhas - cada usuario mantem sua linha
- A query de features gera 8 colunas prontas para ML em uma unica chamada ao banco

### O que concluir

- **Feature engineering em SQL e mais eficiente para dados grandes**: a agregacao acontece no banco, nao na memoria
- **LEFT JOIN e o join mais seguro para ML**: garante que nenhum usuario e perdido (INNER JOIN perde quem nao tem eventos)
- **Window functions sao o "segredo" de SQL avancado**: rankings, medias moveis e lag features ficam triviais

### Conexao com outros notebooks

- As features geradas em SQL alimentam diretamente o pipeline de `4_1_pipeline_ml`
- Window functions para features temporais sao usadas em `3_1_feature_engineering`

## 3. REST APIs e Autenticacao

**Analogia**: Uma API e como um garcom em um restaurante. Voce faz o pedido (request),
o garcom leva a cozinha (servidor), e traz seu prato (response). Voce nao precisa saber
como a cozinha funciona - so precisa saber o que pedir e como pedir.

**Definicao formal**: REST (Representational State Transfer) e um padrao de comunicacao
via HTTP. Os verbos principais sao: GET (buscar), POST (criar), PUT (atualizar),
DELETE (remover). Respostas tipicamente vem em JSON.

### Por que em ML?

APIs sao a fonte de dados externos: clima, cotacoes, redes sociais, geocoding.
Em producao, modelos frequentemente consomem APIs em tempo real para enriquecer features.

In [ ]:
print('\n=== REST APIS ===' )

print('\nEstrutura de requisição:')
print('  GET /users: listar usuários')
print('  GET /users/1: obter usuário 1')
print('  POST /users: criar novo usuário')
print('  PUT /users/1: atualizar usuário 1')
print('  DELETE /users/1: deletar usuário 1')

print('\nAutenticação:')
print('  1. Bearer Token: Authorization: Bearer <token>')
print('  2. API Key: X-API-Key: <key>')
print('  3. Basic Auth: Authorization: Basic <base64(user:pass)>')
print('  4. OAuth 2.0: obter access_token via login')

print('\nExemplos de requisições:')

# Exemplo 1: API pública (OpenWeather)
print('\n1. API pública (sem autenticação):')
print('   GET https://api.open-meteo.com/v1/forecast')
print('   Parâmetros: latitude, longitude, current_weather')

# Requisição simulada
url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude': 40.7128,
    'longitude': -74.0060,
    'current_weather': True
}

try:
    response = requests.get(url, params=params, timeout=5)
    print(f'\nResposta: Status {response.status_code}')
    if response.status_code == 200:
        data = response.json()
        print(f'Temperatura atual: {data["current_weather"]["temperature"]}°C')
except Exception as e:
    print(f'Erro na requisição: {e}')

print('\n2. API com token (exemplo estrutura):')
headers = {
    'Authorization': 'Bearer token_aqui',
    'Content-Type': 'application/json'
}
print(f'Headers: {headers}')


### O que observar

- HTTP GET e o verbo mais comum para coleta de dados (nao modifica nada no servidor)
- Autenticacao via Bearer Token e o padrao mais usado em APIs modernas
- O status code 200 indica sucesso; qualquer outro requer tratamento especifico
- `response.json()` converte a resposta em dicionario Python automaticamente

### O que concluir

- **Sempre verifique o status code antes de processar**: `response.json()` em status 404 causa erro
- **Guarde tokens em variaveis de ambiente, nunca no codigo**: `os.environ['API_KEY']` e o padrao seguro
- **Timeout e obrigatorio**: sem timeout, o request pode travar indefinidamente

### Conexao com outros notebooks

- Dados de APIs enriquecem features em `3_1_feature_engineering` (ex: dados climaticos para previsao de vendas)
- Em `4_4_monitoramento_modelos`, APIs sao usadas para servir predicoes em producao

## 4. JSON Parsing e Normalizacao

**Analogia**: JSON aninhado e como uma caixa dentro de uma caixa dentro de uma caixa.
Para colocar numa tabela (DataFrame), voce precisa "achatar" tudo - desempacotar cada
caixa e espalhar o conteudo em colunas.

**Definicao formal**: `pd.json_normalize()` converte estruturas JSON hierarquicas em
DataFrames planos. O parametro `record_path` indica onde esta a lista de registros, e
`meta` indica quais campos do nivel superior incluir.

### Por que em ML?

APIs reais retornam JSON aninhado (usuario -> perfil -> endereco -> cidade). Sem saber
"achatar" JSON, voce nao consegue criar features a partir de dados de API.

In [ ]:
# JSON Parsing e Normalizacao
print('=== JSON PARSING ===')

# 1. JSON simples
json_simples = [
    {"id": 1, "nome": "Alice", "cidade": "SP"},
    {"id": 2, "nome": "Bob", "cidade": "RJ"},
]
df_simples = pd.DataFrame(json_simples)
print("1. JSON simples -> DataFrame:")
print(df_simples)

# 2. JSON aninhado
json_aninhado = {
    "empresa": "TechCorp",
    "funcionarios": [
        {"nome": "Alice", "perfil": {"idade": 25, "cargo": "DS"}, "skills": ["Python", "SQL"]},
        {"nome": "Bob", "perfil": {"idade": 30, "cargo": "MLE"}, "skills": ["Java", "Spark"]},
        {"nome": "Charlie", "perfil": {"idade": 28, "cargo": "DS"}, "skills": ["R", "SQL"]}
    ]
}

print("\n2. JSON aninhado - estrutura:")
print(f"  Nivel 0: empresa = {json_aninhado['empresa']}")
print(f"  Nivel 1: funcionarios (lista de {len(json_aninhado['funcionarios'])} dicts)")
print(f"  Nivel 2: perfil (dict aninhado), skills (lista)")

# 3. json_normalize para achatar
df_norm = pd.json_normalize(
    json_aninhado['funcionarios'],
    meta=['nome'],
    record_path='skills',
    record_prefix='skill_'
)
print("\n3. json_normalize (record_path='skills'):")
print(df_norm)

# 4. Abordagem alternativa: achatar perfil
df_flat = pd.json_normalize(json_aninhado['funcionarios'])
print("\n4. json_normalize (achatar tudo):")
print(df_flat)

# 5. Caso real: lista de skills como feature
df_flat['n_skills'] = df_flat['skills'].apply(len)
df_flat['has_python'] = df_flat['skills'].apply(lambda x: 'Python' in x).astype(int)
df_flat['has_sql'] = df_flat['skills'].apply(lambda x: 'SQL' in x).astype(int)
print("\n5. Features derivadas de listas aninhadas:")
print(df_flat[['nome', 'perfil.cargo', 'n_skills', 'has_python', 'has_sql']])

### O que observar

- JSON simples (lista de dicts planos) converte direto com `pd.DataFrame()` - nao precisa de normalize
- JSON aninhado requer `json_normalize()` para "achatar" dicts internos (perfil.idade, perfil.cargo)
- Listas dentro de JSON (skills) precisam de tratamento especial: ou `record_path` ou `apply(len)`
- `json_normalize` cria colunas com ponto como separador (perfil.idade) - pode precisar renomear

### O que concluir

- **Explore a estrutura do JSON antes de parsear**: use `json.dumps(data, indent=2)` para visualizar
- **json_normalize resolve 80% dos casos**: so use loop manual quando a estrutura e muito irregular
- **Listas aninhadas viram features**: contagem (n_skills), presenca (has_python), primeiro item, etc.

### Conexao com outros notebooks

- Features derivadas de JSON (n_skills, has_python) sao feature engineering aplicado de `3_1_feature_engineering`
- Em `2_4_acesso_banco_dados`, JSON e usado para configuracao de conexoes

## 5. Coleta com Paginacao

**Analogia**: Imagina pedir 10.000 resultados de uma busca no Google. Ele nao mostra
tudo de uma vez - mostra 10 por pagina. APIs funcionam igual: retornam dados em "paginas"
que voce deve coletar iterativamente.

**Definicao formal**: Paginacao e o mecanismo que APIs usam para dividir respostas grandes.
Tipos: offset/limit (posicao + quantidade), page-based (numero de pagina), cursor-based
(token opaco para proxima pagina).

### Por que em ML?

Datasets reais tem milhoes de registros. Sem paginacao, voce nao consegue coletar tudo.
E sem rate limiting, a API bloqueia voce.

In [ ]:
print('\n=== PAGINAÇÃO EM APIs ===' )

print('\nMétodos de paginação:')
print('  1. Offset/Limit: ?offset=0&limit=10')
print('  2. Page-based: ?page=1&per_page=10')
print('  3. Cursor-based: ?cursor=abc123&limit=10')

print('\nExemplo com JSONPlaceholder (API fake gratuita):')

# Simular resposta da API sem fazer requisição real
all_posts = []
try:
    # Simulamos com dados locais em vez de fazer requisição
    for page in range(1, 6):
        posts = [
            {'userId': (page-1)*10 + j, 'id': (page-1)*10 + j, 'title': f'Post {(page-1)*10 + j}'}
            for j in range(1, 11)
        ]
        all_posts.extend(posts)
        print(f'  Pagina {page}: {len(posts)} posts carregados (simulado)')
except Exception as e:
    print(f'Erro na página: {e}')

print(f'\nTotal de posts coletados: {len(all_posts)}')

if all_posts:
    print('\nPrimeiros registros:')
    import pandas as pd
    df_posts = pd.DataFrame(all_posts[:5])
    print(df_posts[['userId', 'id', 'title']])
else:
    print('Nenhum post coletado (simulação sem requisição real)')

### O que observar

- Cada pagina retorna um lote fixo (10 posts por pagina neste caso)
- O loop para quando: (a) a pagina vem vazia, (b) o status nao e 200, ou (c) atingiu o limite
- `all_posts.extend(posts)` acumula os resultados de todas as paginas em uma unica lista

### O que concluir

- **Sempre implemente paginacao desde o inicio**: APIs com muitos dados EXIGEM isso
- **Defina um limite maximo de paginas**: sem limite, um bug pode fazer requests infinitos
- **Valide duplicatas apos coleta**: APIs bugadas podem retornar dados repetidos entre paginas

### Conexao com outros notebooks

- Os dados paginados sao o input bruto para limpeza em `2_2_eda_completa`
- Em producao (`4_4_monitoramento_modelos`), coleta paginada alimenta pipelines batch

## 6. Rate Limiting e Retry Logic

**Analogia**: Rate limiting e como uma fila de banco - voce nao pode atender todos ao
mesmo tempo. Se voce tentar fazer requests demais, a API diz "espere" (erro 429).
Retry com backoff exponencial e como esperar 1 segundo, depois 2, depois 4... ate a
API aceitar de novo.

**Definicao formal**: Rate limiting restringe o numero de requests por intervalo de tempo.
Backoff exponencial e uma estrategia de retry onde o tempo de espera dobra a cada tentativa:
wait = base * 2^attempt.

### Por que em ML?

Em coleta de dados para treino, voce pode precisar de milhoes de requests. Sem rate
limiting, a API te bloqueia. Sem retry, um erro temporario descarta horas de coleta.

In [ ]:
import time
from functools import wraps

print('\n=== RATE LIMITING E RETRY ===' )

print('\nProblema: APIs têm limites de requisições')
print('  - Twitter: 450 req/15 min')
print('  - GitHub: 60 req/min (sem auth)')

print('\nSolução: Implementar retry logic com backoff')

def retry_with_backoff(max_retries=3, base_wait=1):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except requests.exceptions.RequestException as e:
                    if attempt < max_retries - 1:
                        wait_time = base_wait * (2 ** attempt)
                        print(f'Tentativa {attempt + 1} falhou. Aguardando {wait_time}s...')
                        time.sleep(wait_time)
                    else:
                        print(f'Falhou após {max_retries} tentativas')
                        raise
        return wrapper
    return decorator

@retry_with_backoff(max_retries=3)
def fetch_api(url):
    response = requests.get(url, timeout=5)
    response.raise_for_status()  # Raise se status != 2xx
    return response.json()

print('\nDecorador @retry_with_backoff criado')
print('Uso: aplicar a funções que chamam APIs')

print('\nRate limiter com espera:')

class RateLimiter:
    def __init__(self, max_requests_per_second=1):
        self.max_requests = max_requests_per_second
        self.min_interval = 1.0 / max_requests_per_second
        self.last_request_time = 0
    
    def wait(self):
        elapsed = time.time() - self.last_request_time
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self.last_request_time = time.time()

limiter = RateLimiter(max_requests_per_second=2)
print('Rate limiter: 2 requisições por segundo')


### O que observar

- O decorator `@retry_with_backoff` e reutilizavel em qualquer funcao que faca requests
- Backoff exponencial: 1s, 2s, 4s - evita sobrecarregar a API com retentativas rapidas
- O RateLimiter controla a frequencia de requests mesmo quando as respostas sao rapidas

### O que concluir

- **Implemente retry desde o dia 1**: nao espere o erro acontecer em producao
- **Backoff exponencial e o padrao da industria**: usado por AWS, Google Cloud, etc.
- **Combine rate limiter + retry**: rate limiter previne 429; retry lida com erros temporarios

### Conexao com outros notebooks

- Em `4_4_monitoramento_modelos`, rate limiting protege APIs de predicao em producao
- Robustez de coleta e pre-requisito para pipelines confiaveis em `4_1_pipeline_ml`

## 7. Tratamento de Erros em APIs

**Analogia**: Quando voce liga para um numero e ninguem atende, voce nao desiste na
primeira tentativa - voce espera e tenta de novo. E se der ocupado (429), voce espera
mais. Tratamento de erros em APIs e exatamente isso.

**Definicao formal**: Codigos HTTP indicam o resultado: 2xx (sucesso), 4xx (erro do
cliente), 5xx (erro do servidor). Uma funcao robusta de coleta trata cada caso
especificamente e decide se faz retry ou aborta.

### Por que em ML?

Um pipeline de dados que falha silenciosamente e pior que um que da erro explicito.
Sem tratamento de erros, voce pode treinar um modelo com metade dos dados sem perceber.

In [ ]:
print('\n=== TRATAMENTO DE ERROS ===' )

print('\nCódigos HTTP comuns:')
print('  2xx: Sucesso')
print('  4xx: Erro do cliente (autenticação, validação)')
print('  5xx: Erro do servidor')

def safe_api_request(url, params=None, timeout=5):
    try:
        response = requests.get(url, params=params, timeout=timeout)
        
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 401:
            print('Erro 401: Autenticação falhou')
        elif response.status_code == 404:
            print('Erro 404: Recurso não encontrado')
        elif response.status_code == 429:
            print('Erro 429: Rate limit excedido')
        elif response.status_code >= 500:
            print('Erro 5xx: Problema no servidor')
        else:
            print(f'Erro {response.status_code}: {response.reason}')
        
        return None
    
    except requests.exceptions.Timeout:
        print('Timeout: Servidor demorou demais')
    except requests.exceptions.ConnectionError:
        print('Erro de conexão')
    except json.JSONDecodeError:
        print('Erro: Resposta não é JSON válido')
    except Exception as e:
        print(f'Erro inesperado: {e}')
    
    return None

print('\nFunção safe_api_request criada')
print('Trata: timeout, conexão, JSON inválido, erros HTTP')


### O que observar

- Cada codigo de erro tem um tratamento diferente: 401 (autenticacao), 404 (nao existe), 429 (rate limit), 5xx (servidor)
- `requests.exceptions.Timeout` e tratado separadamente porque o servidor pode estar lento, nao com erro
- A funcao retorna None em caso de erro - quem chama decide o que fazer

### O que concluir

- **Nunca ignore status codes**: `response.json()` sem checar status e um bug esperando acontecer
- **Timeout deve ser sempre explicito**: o default de requests e *sem timeout* (pode travar para sempre)
- **Logging e essencial**: em producao, voce precisa saber *qual* request falhou e *por que*

### Conexao com outros notebooks

- Robustez de API e essencial para features em tempo real de `4_4_monitoramento_modelos`
- O padrao try/except com logging e usado em todo `4_1_pipeline_ml`

## 8. Web Scraping Basico

**Analogia**: Web scraping e como ler um jornal e anotar os dados que interessam. A
pagina HTML e o jornal; BeautifulSoup e a lente de aumento que te ajuda a encontrar
e extrair tabelas, links e textos.

**Definicao formal**: Web scraping e a extracao automatizada de dados de paginas HTML.
BeautifulSoup faz parsing do HTML e permite buscar elementos por tag, classe, id.
Pandas `read_html()` extrai tabelas automaticamente.

### Por que em ML?

Quando os dados nao estao em API ou banco, scraping e a unica opcao: precos de
concorrentes, reviews de produtos, dados publicos governamentais. Mas respeite
robots.txt e termos de servico.

In [ ]:
print('\n=== WEB SCRAPING ===' )

from bs4 import BeautifulSoup

print('\nAviso legal: respeite robots.txt e terms of service')

html = '''
<html>
    <body>
        <table>
            <tr><th>País</th><th>Capital</th><th>População</th></tr>
            <tr><td>Brasil</td><td>Brasília</td><td>215M</td></tr>
            <tr><td>USA</td><td>Washington</td><td>331M</td></tr>
            <tr><td>China</td><td>Pequim</td><td>1.4B</td></tr>
        </table>
    </body>
</html>
'''

soup = BeautifulSoup(html, 'html.parser')

print('\n1. Extrair tabela:')
table = soup.find('table')
rows = table.find_all('tr')

data = []
for row in rows[1:]:  # Skip header
    cols = row.find_all('td')
    data.append([col.text.strip() for col in cols])

df_scraped = pd.DataFrame(data, columns=['País', 'Capital', 'População'])
print(df_scraped)

print('\n2. Extrair links:')
links_html = '<a href="/page1">Page 1</a><a href="/page2">Page 2</a>'
soup_links = BeautifulSoup(links_html, 'html.parser')
for link in soup_links.find_all('a'):
    print(f'  {link.text}: {link.get("href")}')


### O que observar

- BeautifulSoup converte HTML em uma arvore navegavel com `find()` e `find_all()`
- Tabelas HTML mapeiam diretamente para DataFrames - rows/cols -> linhas/colunas
- `soup.find_all('a')` extrai todos os links da pagina - util para crawling

### O que concluir

- **Sempre verifique robots.txt e termos de servico antes de scrapear**: scraping ilegal pode ter consequencias serias
- **Prefira APIs quando disponiveis**: scraping e fragil (muda o HTML, quebra o scraper); APIs sao estaveis
- **Use delays entre requests**: scraping agressivo pode derrubar o site e bloquear seu IP

### Conexao com outros notebooks

- Dados scrapeados passam pelo mesmo pipeline de limpeza de `2_2_eda_completa`
- Em `3_1_feature_engineering`, texto scrapeado pode ser transformado em features (TF-IDF, embeddings)

## 9. Exercicios Praticos

### Exercicio 1: Feature Engineering via SQL

Usando o banco SQLite ja criado (tabelas users e events), escreva queries para criar
3 features uteis para prever churn (usuarios que param de usar o servico).

In [ ]:
# TAREFA DO ALUNO: Exercicio 1 - Features de Churn via SQL
# Usando conn (ja conectado) com tabelas users e events

# Feature 1: dias desde o ultimo evento (recencia)
# TAREFA DO ALUNO: query SQL que calcula MAX(event_date) por usuario

# Feature 2: total gasto por usuario
# TAREFA DO ALUNO: query SQL que calcula SUM(value) por usuario

# Feature 3: ratio de compras vs logins
# TAREFA DO ALUNO: query SQL que calcula n_purchases / n_events por usuario

df_churn_features = None  # TAREFA DO ALUNO: pd.read_sql_query(...)
print(df_churn_features)

In [ ]:
# SOLUCAO - Exercicio 1
print("=== FEATURES DE CHURN VIA SQL ===")

df_churn_features = pd.read_sql_query('''
    SELECT
        u.id,
        u.name,
        -- Feature 1: recencia (dias desde ultimo evento)
        MAX(e.event_date) as ultimo_evento,
        julianday('2023-04-01') - julianday(MAX(e.event_date)) as dias_desde_ultimo,
        -- Feature 2: total gasto
        COALESCE(SUM(e.value), 0) as total_gasto,
        -- Feature 3: ratio compras/eventos
        ROUND(
            CAST(SUM(CASE WHEN e.event_type = 'purchase' THEN 1 ELSE 0 END) AS FLOAT) /
            NULLIF(COUNT(e.event_id), 0),
            2
        ) as ratio_compras,
        -- Bonus: numero total de eventos
        COUNT(e.event_id) as n_eventos
    FROM users u
    LEFT JOIN events e ON u.id = e.user_id
    GROUP BY u.id, u.name
    ORDER BY dias_desde_ultimo DESC
''', conn)

print(df_churn_features)
print("\nInterpretacao:")
print("  - Eve (id=5) tem maior recencia e apenas 1 evento -> alto risco de churn")
print("  - Diana (id=4) gastou mais (200) mas pouca atividade -> risco medio")
print("  - Bob (id=2) tem ratio alto de compras (0.67) -> engajado, baixo risco")

### Exercicio 2: Coleta Paginada com Validacao

Colete dados da API JSONPlaceholder com paginacao e valide a qualidade dos dados:
contagem total, duplicatas, nulos, tipos.

In [ ]:
# TAREFA DO ALUNO: Exercicio 2 - Coleta Paginada com Validacao
import time

url_base = 'https://jsonplaceholder.typicode.com/posts'
all_data = []

# TAREFA DO ALUNO: Loop de paginacao (5 paginas, 10 posts cada)
# TAREFA DO ALUNO: Adicionar rate limiting (1 segundo entre requests)
# TAREFA DO ALUNO: Tratar erros (timeout, status code)

# TAREFA DO ALUNO: Converter para DataFrame
# df_coletado = pd.DataFrame(all_data)

# TAREFA DO ALUNO: Validacao
# print(f"Total coletado: {len(df_coletado)}")
# print(f"Duplicatas: {df_coletado.duplicated().sum()}")
# print(f"Nulos: {df_coletado.isnull().sum().sum()}")

In [ ]:
# SOLUCAO - Exercicio 2
import time

url_base = 'https://jsonplaceholder.typicode.com/posts'

print('=== COLETA PAGINADA COM VALIDACAO ===')

# Simular coleta paginada sem fazer requisições reais
all_posts = []
try:
    for page in range(1, 6):
        # Simular resposta
        posts = [
            {'userId': (page-1)*10 + j, 'id': (page-1)*10 + j, 'title': f'Post {(page-1)*10 + j}'}
            for j in range(1, 11)
        ]
        all_posts.extend(posts)
        print(f'  Pagina {page}: {len(posts)} posts')
        time.sleep(0.1)
except Exception as e:
    print(f'Erro: {e}')

print(f'\n=== VALIDACAO ===')
print(f'Total coletado: {len(all_posts)}')

if all_posts:
    import pandas as pd
    df = pd.DataFrame(all_posts)
    print(f'Colunas: {list(df.columns)}')
    print(f'Duplicatas: {df.duplicated().sum()}')
    print(f'Nulos: {df.isnull().sum().sum() / (len(df) * len(df.columns)) * 100:.1f}%')
    print(f'Dtypes:\n{df.dtypes}')
    print(f'\nAmostra:\n{df.head()}')
else:
    print('Nenhum dado coletado (simulação sem requisição real)')

### Exercicio 3: JSON Parsing de API Real

Parse o JSON aninhado abaixo (simulando resposta de API) e crie um DataFrame
com features uteis para ML.

In [ ]:
# TAREFA DO ALUNO: Exercicio 3 - Parse de JSON Aninhado
json_api = {
    "status": "ok",
    "total": 3,
    "data": [
        {"user": {"id": 1, "name": "Alice"}, "orders": [{"item": "A", "price": 10}, {"item": "B", "price": 20}]},
        {"user": {"id": 2, "name": "Bob"}, "orders": [{"item": "C", "price": 30}]},
        {"user": {"id": 3, "name": "Charlie"}, "orders": []}
    ]
}

# TAREFA DO ALUNO: Extrair lista de usuarios com suas features
# TAREFA DO ALUNO: Para cada usuario: id, name, n_orders, total_spent, avg_order
# Dica: itere sobre json_api['data'] e construa um dicionario por usuario

df_parsed = None  # TAREFA DO ALUNO
print(df_parsed)

In [ ]:
# SOLUCAO - Exercicio 3
json_api = {
    "status": "ok",
    "total": 3,
    "data": [
        {"user": {"id": 1, "name": "Alice"}, "orders": [{"item": "A", "price": 10}, {"item": "B", "price": 20}]},
        {"user": {"id": 2, "name": "Bob"}, "orders": [{"item": "C", "price": 30}]},
        {"user": {"id": 3, "name": "Charlie"}, "orders": []}
    ]
}

print("=== PARSE DE JSON ANINHADO ===")

# Abordagem: iterar e construir features
records = []
for entry in json_api['data']:
    user = entry['user']
    orders = entry['orders']
    records.append({
        'user_id': user['id'],
        'name': user['name'],
        'n_orders': len(orders),
        'total_spent': sum(o['price'] for o in orders),
        'avg_order': sum(o['price'] for o in orders) / len(orders) if orders else 0,
        'has_orders': int(len(orders) > 0)
    })

df_parsed = pd.DataFrame(records)
print(df_parsed)

print("\nAlternativa com json_normalize:")
df_alt = pd.json_normalize(json_api['data'])
print(df_alt)
print("\nNota: json_normalize nao resolve listas internas (orders) automaticamente")
print("Para listas, o loop manual e mais controlavel.")

### O que observar nos exercicios

- O Exercicio 1 mostra que SQL pode gerar features completas sem sair do banco - mais eficiente que Pandas para dados grandes
- O Exercicio 2 demonstra que coleta robusta exige paginacao + rate limiting + validacao - nao basta fazer GET
- O Exercicio 3 revela que JSON aninhado real exige iteracao manual para extrair features - json_normalize nem sempre basta

### O que concluir dos exercicios

- **SQL e a linguagem mais importante para data engineering**: saber SQL avancado e diferencial de mercado
- **Coleta de dados e um pipeline, nao uma chamada**: request -> validate -> store -> repeat
- **JSON parsing e uma skill subestimada**: a maioria dos dados modernos vem em JSON aninhado

### Conexao com outros notebooks

- As features SQL do Exercicio 1 sao o input para `4_1_pipeline_ml`
- A coleta paginada do Exercicio 2 alimenta EDA em `2_2_eda_completa`

### O que observar no panorama geral

- SQL e APIs sao as duas fontes primarias de dados em producao - CSV e so para prototipagem
- O fluxo completo e: SQL/API (fonte) -> JSON/DataFrame (formato) -> EDA -> Features -> Modelo

### O que concluir do panorama geral

- **Dominar SQL e APIs e mais valioso que dominar mais um algoritmo de ML**: dados bons com modelo simples supera dados ruins com modelo complexo
- **Robustez na coleta previne problemas downstream**: dados incompletos geram modelos ruins

### Conexao com outros notebooks

- Este notebook e o "ponto de entrada" de dados para todo o pipeline que comeca em `2_2_eda_completa`
- Em `2_4_acesso_banco_dados`, voce aprofunda conexoes com bancos reais (PostgreSQL, MySQL)

## 10. Erros Comuns e Armadilhas

### Erro 1: Esquecer GROUP BY em queries com agregacao
`SELECT user_id, COUNT(*) FROM events` sem GROUP BY e ambiguo e retorna erro (ou resultado
errado dependendo do banco). Sempre inclua GROUP BY para cada coluna nao-agregada.

### Erro 2: JOIN cartesiano (sem condicao ON)
`SELECT * FROM users, events` sem ON gera produto cartesiano: 5 usuarios x 10 eventos = 50
linhas! Sempre especifique a condicao de join.

### Erro 3: Ignorar status code da API
`response.json()` com status 404 ou 500 gera erro ou retorna dados inesperados. Sempre
verifique `response.status_code == 200` antes de processar.

### Erro 4: Nao implementar timeout em requests
`requests.get(url)` sem `timeout` pode travar indefinidamente se o servidor nao responder.
Sempre use `timeout=5` (ou valor apropriado).

### Erro 5: Token de API hardcoded no codigo
`headers = {'Authorization': 'Bearer meu_token_secreto'}` no notebook e um risco de seguranca.
Use `os.environ['API_TOKEN']` e arquivos `.env` para armazenar credenciais.

### Erro 6: Scraping sem respeitar robots.txt
Acessar paginas proibidas por robots.txt pode ter consequencias legais. Sempre verifique
`site.com/robots.txt` antes de scrapear. Prefira APIs quando disponiveis.

### Erro 7: Nao validar dados apos coleta
Coletar 10.000 registros via API sem verificar duplicatas, nulos ou tipos e um convite
para problemas na modelagem. Sempre valide imediatamente apos a coleta.

## 11. Resumo e Conexoes

### Hierarquia de Conceitos

```
FONTES DE DADOS
    |
    |---> SQL (bancos relacionais)
    |      |---> SELECT, WHERE, JOIN, GROUP BY
    |      |---> Window Functions (RANK, LAG, SUM OVER)
    |      |---> Feature engineering no banco
    |      |---> pd.read_sql_query() -> DataFrame
    |
    |---> APIs REST (servicos web)
    |      |---> GET, POST, headers, autenticacao
    |      |---> JSON parsing e json_normalize
    |      |---> Paginacao (offset, page, cursor)
    |      |---> Rate limiting + retry + backoff
    |
    |---> Web Scraping (paginas HTML)
    |      |---> BeautifulSoup (find, find_all)
    |      |---> robots.txt e etica
    |
    v
DADOS NO PANDAS -> EDA -> FEATURES -> MODELO
```

### Tabela de Conexoes

| Fonte | Ferramenta | Notebook futuro |
|-------|-----------|-----------------|
| SQL basico | SELECT, WHERE | `2_4_acesso_banco_dados` (PostgreSQL) |
| SQL features | GROUP BY, Window | `3_1_feature_engineering` |
| APIs REST | requests, JSON | `4_4_monitoramento` (serving) |
| Rate limiting | retry, backoff | `4_1_pipeline_ml` (robustez) |
| Web scraping | BeautifulSoup | `3_1_feature_engineering` (texto) |
| JSON parsing | json_normalize | `2_2_eda_completa` |

### Checklist de Competencias

- [ ] Sei escrever SELECT, WHERE, GROUP BY, ORDER BY em SQL
- [ ] Sei fazer JOINs (LEFT, INNER) e entendo a diferenca
- [ ] Sei usar window functions (RANK, ROW_NUMBER)
- [ ] Sei fazer requests GET com autenticacao (Bearer Token)
- [ ] Sei parsear JSON aninhado com json_normalize e loops
- [ ] Sei implementar paginacao para coleta em larga escala
- [ ] Sei usar retry com backoff exponencial
- [ ] Sei tratar erros HTTP (status codes, timeout)
- [ ] Sei fazer web scraping basico com BeautifulSoup

### Proximos Passos

1. **`2_4_acesso_banco_dados`**: Conectar a bancos reais (PostgreSQL, MySQL)
2. **`3_1_feature_engineering`**: Usar dados coletados para criar features avancadas
3. **`4_1_pipeline_ml`**: Integrar coleta no pipeline completo de ML